In [ ]:
!pip install eyed3 pyAudioAnalysis
!apt-get install ffmpeg -y

In [ ]:
import numpy as np
import subprocess
from pyAudioAnalysis import audioBasicIO, ShortTermFeatures

In [ ]:
video_path = '/content/Video Interviewing Example.mp4'
audio_output_path = "extracted_audio.wav"

# extract audio to extracted_audio.wav
subprocess.run(["ffmpeg", "-y", "-i", video_path, "-q:a", "0", "-map", "a", audio_output_path])

# Fs is the sampling rate (how many discrete numbers he capture each second) -Hz-
# x is the actual array data for wave, numpay array (should be Hz * num of seconds)
# audioBasicIO auto detect a good number for sampling rate (Fs) from the video
Fs, x = audioBasicIO.read_audio_file(audio_output_path)

# pyAudioAnalysis library expect mono voice
x = audioBasicIO.stereo_to_mono(x)

print(f"sampling rate: {Fs} Hz")

sampling rate: 44100 Hz


In [ ]:
def calculate_speech_energy(audio_signal, sample_rate):
    """
    Calculates energy statistics for a given audio signal.

    Args:
        audio_signal (np.ndarray): The mono audio signal (x).
        sample_rate (int): The sampling frequency (Fs).

    Returns:
        dict: Average, Max, Min, and Std of the speech energy.
    """

    # 50ms window, 25ms step
    # overlap to catch all features in voice

    win = int(0.050 * sample_rate)
    step = int(0.025 * sample_rate)


    # the work on the audio:
    # f_names is the names of features ['energy', 'entropy',..... etc]
    # F is 2D matrix for numeric values for each window for the features
    F, f_names = ShortTermFeatures.feature_extraction(audio_signal, sample_rate, win, step)


    # we need energy only for this task
    # extract the energy column only
    try:
        energy_idx = f_names.index('energy')
    except ValueError:
        energy_idx = 1

    # energy values
    energy_frames = F[energy_idx, :].astype(float)

    # final calculations
    stats = {
        "average_energy": float(np.mean(energy_frames)),
        "max_energy": float(np.max(energy_frames)),
        "min_energy": float(np.min(energy_frames)),
        "std_energy": float(np.std(energy_frames))
    }

    return stats

In [ ]:
energy_results = calculate_speech_energy(x, Fs)
print("Energy Analysis Results:")
for key, value in energy_results.items():
    print(f"  {key}: {value:.6f}")

Energy Analysis Results:
  average_energy: 0.002212
  max_energy: 0.127547
  min_energy: 0.000000
  std_energy: 0.005692
